[目录](./table_of_contents.ipynb)

# 非线性滤波

In [ ]:
%matplotlib inline

In [ ]:
#设置书的格式
import book_format
book_format.set_style()

## 引言

我们开发的卡尔曼滤波器使用线性方程，因此该滤波器只能处理线性问题。但是，世界是非线性的，因此我们一直在研究的经典滤波器在很多情况下的效用非常有限。

过程模型中可能存在非线性。假设我们想要跟踪一架穿过大气层的物体。物体的加速度取决于它所遇到的阻力。阻力取决于空气密度，而空气密度随着海拔的升高而降低。在一维中，这可以用非线性微分方程来模拟

$$\ddot x = \frac{0.0034ge^{-x/22000}\dot x^2}{2\beta} - g$$

非线性的第二个来源来自测量。例如，雷达测量物体的斜距，我们通常感兴趣的是飞机在地面上的位置。我们使用勾股定理得到非线性方程：

$$x=\sqrt{\mathtt{slant}^2 - \mathtt{altitude}^2}$$

这些事实并没有被卡尔曼滤波器的早期采用者忽视。卡尔曼博士发表论文后不久，人们就开始研究如何扩展卡尔曼滤波器以解决非线性问题。

可以说，我们唯一知道如何解决的方程式是 $\mathbf{Ax}=\mathbf{b}$。我们只真正了解如何进行线性代数。我可以给你任何线性方程组，你可以解决它或证明它没有解。

任何接受过数学或物理正式教育的人都花了数年时间学习各种解决积分、微分方程等的分析方法。然而，即使是微不足道的物理系统也会产生无法解析求解的方程。我可以将你能够积分的方程插入一个 $\log$ 项，并使其无法求解。这引出了物理学家们关于“假设一个在真空中的摩擦系数为零的球体奶牛...”的笑话。除非进行极端简化，大多数物理问题都没有解析解。

那么我们如何在计算机上模拟飞机气流、预测天气或使用卡尔曼滤波器跟踪导弹呢？我们退而求其次，回到我们熟悉的领域：$\mathbf{Ax}=\mathbf{b}$。我们找到某种方法将问题线性化，转化为一组线性方程，并使用线性代数软件包计算近似解。

非线性问题的线性化会导致不精确的答案，在递归算法中（如卡尔曼滤波器或天气跟踪系统），这些小错误有时会在每一步强化，迅速导致算法输出无意义的结果。

我们即将面临的是一个困难的问题。不再有一个明显的、正确的、数学上最优的解决方案。我们将使用近似方法，引入误差到我们的计算中，并且我们将永远与*发散*的滤波器作斗争，即数值误差压倒解决方案的滤波器。

在本章的其余部分，我将说明非线性卡尔曼滤波器面临的具体问题。只有理解问题中的非线性特定问题后，你才能设计出一个滤波器。随后的章节将教你如何设计和实现不同类型的非线性滤波器。

## 非线性的问题


卡尔曼滤波器的数学部分非常美妙，部分原因在于高斯方程非常特殊。它是非线性的，但是当我们将它们相加和相乘时，我们会得到另一个高斯方程作为结果。这非常罕见。$\sin{x}*\sin{y}$ 不会产生 $\sin$ 作为输出。

我所说的线性可能很明显，但有一些微妙之处。数学要求有两个：

* 可加性：$f(x+y) = f(x) + f(y)$
* 齐次性：$f(ax) = af(x)$

这让我们说，线性系统被定义为其输出与所有输入的总和成线性比例关系的系统。这意味着，如果输入为零，则输出也必须为零，才能保证线性。考虑一个音频放大器 - 如果我唱歌，你开始说话，输出应该是我们声音（输入）的总和乘以放大器增益。但是，如果放大器输出非零信号（如嗡嗡声），则加法关系不再成立。这是因为线性要求 $amp(voice) = amp(voice + 0)$。这显然应该给出相同的输出，但如果 amp(0) 是非零的，则

$$
\begin{aligned}
amp(voice) &= amp(voice + 0) \\
&= amp(voice) + amp(0) \\
&= amp(voice) + 非零值
\end{aligned}
$$

这显然是不合理的。因此，一个表面上线性的方程，例如

$$L(f(t)) = f(t) + 1$$

并不是线性的，因为 $L(0) = 1$。要小心！

## 问题的直观看法

我特别喜欢以下的解决问题方式，这是我从Dan Simon的*Optimal State Estimation* [[1]](#[1])中借鉴过来的。考虑一个跟踪问题，我们得到了一个目标的距离和方位角，我们想要跟踪它的位置。报告的距离为50公里，报告的角度为90度。假设距离和角度的误差都是高斯分布的。在无限次测量中，位置的期望值是多少？

我一直推荐使用直觉来获得洞察力，那么我们来看看它在这个问题上的表现。我们可能会推断，由于距离的平均值将是50公里，角度的平均值将是90度，因此答案将是x=0公里，y=50公里。

让我们绘制它并找出答案。这里有3000个点，以0.4公里的距离和0.35弧度的正常分布进行绘制。我们计算所有位置的平均值，并将其显示为星形。我们的直觉以一个大圆显示。

In [ ]:
import numpy as np
from numpy.random import randn
import matplotlib.pyplot as plt

N = 5000
a = np.pi/2. + (randn(N) * 0.35)
r = 50.0     + (randn(N) * 0.4)
xs = r * np.cos(a)
ys = r * np.sin(a)

plt.scatter(xs, ys, label='传感器', color='k', 
            alpha=0.4, marker='.', s=1)
xmean, ymean = sum(xs) / N, sum(ys) / N
plt.scatter(0, 50, c='k', marker='o', s=200, label='直觉')
plt.scatter(xmean, ymean, c='r', marker='*', s=200, label='均值')
plt.axis('equal')
plt.legend();

我们可以看到，我们的直觉失败了，因为问题的非线性导致所有错误都偏向一个方向。这种偏差在许多迭代中会导致卡尔曼滤波器发散。即使它不发散，解决方案也不会是最优的。在非线性问题上应用线性近似会产生不准确的结果。

## 非线性函数对高斯分布的影响

高斯分布不满足任意非线性函数的封闭性。回想一下卡尔曼滤波器的方程——在每次演进中，我们通过过程函数将表示状态的高斯分布传递到时间 $k$ 的高斯分布。我们的过程函数总是线性的，因此输出始终是另一个高斯分布。让我们在图表上看一下。我将采用任意高斯分布，并通过函数 $f(x)=2x+1$ 将其传递，并绘制结果。我们知道如何以解析的方式做到这一点，但让我们使用采样。我将生成 500,000 个服从正态分布的点，将它们通过 $f(x)$ 传递，并绘制结果。我这样做是因为下一个例子将是非线性的，我们将无法以解析的方式计算它。

In [ ]:
from numpy.random import normal

data = normal(loc=0., scale=1., size=500000)
plt.hist(2*data + 1, 1000);

这是一个毫不意外的结果。将高斯函数通过$f(x)=2x+1$传递的结果是另一个以1为中心的高斯函数。让我们一次性查看输入、非线性函数和输出。

In [ ]:
from kf_book.book_plots import set_figsize, figsize
from kf_book.nonlinear_plots import plot_nonlinear_func

def g1(x):
    return 2*x+1

plot_nonlinear_func(data, g1)

> 我在*Supporting_Notebooks*文件夹中的*Computing_and_Plotting_PDFs*笔记本中解释了如何绘制高斯函数等内容。你也可以在这里[在线阅读](https://github.com/rlabbe/Kalman-and-Bayesian-Filters-in-Python/blob/master/Supporting_Notebooks/Computing_and_plotting_PDFs.ipynb)[1]。

标记为“Input”的图是原始数据的直方图。这些数据经过函数 $f(x)=2x+1$ 处理后显示在左下角的图表中。红线显示了一个值 $x=0$ 如何通过函数。每个输入值都通过相同的方式传递到右侧的输出函数中。对于输出，我通过取所有点的平均值来计算均值，并用虚线蓝线绘制结果。实线蓝线显示点 $x=0$ 的实际均值。输出看起来像一个高斯分布，实际上就是一个高斯分布。我们可以看到输出的方差比输入的方差大，平均值已从 0 移动到 1，这是我们根据传输函数 $f(x)=2x+1$ 所期望的。$2x$ 影响方差，$+1$ 移动平均值。用虚线蓝线表示的计算出的均值几乎等于实际均值。如果我们在计算中使用更多点，我们可以无限接近实际均值。

将其转换为实际值。

现在让我们来看一个非线性函数，并看看它如何影响概率分布。

In [ ]:
def g2(x):
    return (np.cos(3*(x/2 + 0.7))) * np.sin(0.3*x) - 1.6*x

plot_nonlinear_func(data, g2)

这个结果可能会让你有些惊讶。这个函数看起来"相当"线性，但输出的概率分布与高斯分布完全不同。 回想一下两个单变量高斯分布相乘的方程式：

$$\begin{aligned}
\mu &=\frac{\sigma_1^2 \mu_2 + \sigma_2^2 \mu_1} {\sigma_1^2 + \sigma_2^2} \\
\sigma &= \frac{1}{\frac{1}{\sigma_1^2} + \frac{1}{\sigma_2^2}}
\end{aligned}$$

这些方程式不适用于非高斯分布，当然也不适用于上面的"输出"图表所示的概率分布。

这里有另一种通过散点图来查看相同数据的方法。

In [ ]:
N = 30000
plt.subplot(121)
plt.scatter(data[:N], range(N), alpha=.1, s=1.5)
plt.title('输入')
plt.subplot(122)
plt.title('输出')
plt.scatter(g2(data[:N]), range(N), alpha=.1, s=1.5);

原始数据明显是高斯分布的，但是通过 `g2(x)` 处理后的数据不再是正态分布的。在 -3 附近有一条粗线，而点在该线的两侧分布不均匀。如果你将其与前面图表中标有“输出”标签的概率密度函数进行比较，你应该能够看出 pdf 形状如何与 `g(data)` 的分布相匹配。

想想这对前一章节中的卡尔曼滤波器算法意味着什么。所有的方程都假设通过过程函数传递的高斯分布会导致另一个高斯分布。如果这不是真的，那么卡尔曼滤波器的所有假设和保证就不成立了。我们来看看当我们将输出再次通过函数传递时会发生什么，模拟卡尔曼滤波器的下一步时间步骤。

In [ ]:
y = g2(data)
plot_nonlinear_func(y, g2)

如上所示，概率函数与原始高斯函数相比进一步扭曲。然而，图形在x=0附近仍然有些对称，我们看看均值是多少。

In [ ]:
print('输入均值和方差：%.4f，%.4f' % 
      (np.mean(data), np.var(data)))
print('输出均值和方差：%.4f，%.4f' % 
      (np.mean(y), np.var(y)))

让我们将其与通过（-2，3）和（2，-3）的线性函数进行比较，该函数非常接近我们绘制的非线性函数。使用一条直线的方程，我们有

$$m=\frac{-3-3}{2-(-2)}=-1.5$$

In [ ]:
def g3(x): 
    return -1.5 * x

plot_nonlinear_func(data, g3)
out = g3(data)
print('输出均值和方差：%.4f，%.4f' % 
      (np.mean(out), np.var(out)))

虽然输出的形状非常不同，但每个输出的均值和方差几乎相同。这可能会让我们推断，如果非线性方程“接近于”线性，那么我们可以忽略这个问题。为了测试这一点，我们可以进行多次迭代，然后比较结果。

In [ ]:
out = g3(data)
out2 = g2(data)

for i in range(10):
    out = g3(out)
    out2 = g2(out2)
print('线性输出的均值，方差：%.4f，%.4f' % 
      (np.average(out), np.std(out)**2))
print('非线性输出的均值，方差：%.4f，%.4f' % 
      (np.average(out2), np.std(out2)**2))

不幸的是，非线性版本不稳定。它与平均值0的差距显著，方差大了半个数量级。

我通过使用非常接近直线的函数来最小化这个问题。如果函数是 $y(x)=-x^2$ 会发生什么？

In [ ]:
def g3(x): 
    return -x*x

data = normal(loc=1, scale=1, size=500000)
plot_nonlinear_func(data, g3)

尽管在$x=1$处曲线平滑且相当直接，输出的概率分布看起来并不像高斯分布，而且计算出的输出平均值与直接计算出的值相差很大。这不是一个不寻常的函数——一个弹道物体沿着一条抛物线运动，这是你的滤波器需要处理的非线性。如果你还记得，我们曾试图追踪一个球，结果失败惨重。这张图应该让你了解为什么滤波器表现得如此糟糕。

## 一个二维的例子

很难观察概率分布并推断出滤波器中会发生什么。因此，让我们考虑用雷达跟踪飞机。估计值可能具有以下协方差：

In [ ]:
import kf_book.nonlinear_internal as nonlinear_internal

nonlinear_internal.plot1()

当我们试图对这个问题进行线性化时会发生什么？雷达给我们提供了飞机到达的距离。假设雷达直接位于飞机下方（x=10），下一次测量显示飞机距离我们3英里（y=3）。可能与该测量匹配的位置形成了一个半径为3英里的圆，如下所示。

In [ ]:
nonlinear_internal.plot2()

通过检查，我们可以看出飞机的可能位置在x=11.4，y=2.7附近，因为这是协方差椭圆和范围测量重叠的位置。但是，该范围测量是非线性的，因此我们必须对其进行线性化。我们还没有涵盖这个话题，但是扩展卡尔曼滤波器将在线性化飞机的最后位置（10,2）处。在x=10处，范围测量为y=3，因此我们在该点进行线性化。

In [ ]:
nonlinear_internal.plot3()

现在我们有了一个问题的线性表示（真正的一条直线），我们可以解决它。不幸的是，您可以看到线和协方差椭圆的交点离实际飞机位置很远。

In [ ]:
nonlinear_internal.plot4()

这种错误经常导致灾难性的结果。这个估计的误差很大。但是在滤波器的下一个创新中，将使用这个非常糟糕的估计来线性化下一个雷达测量，因此下一个估计很可能比这个估计差得多。在仅仅几次迭代之后，卡尔曼滤波器就会发散，并开始产生与现实无关的结果。

这个协方差椭圆覆盖了好几英里。我夸大了它的大小，以说明高度非线性系统的困难。在实际雷达跟踪问题中，非线性通常并不那么严重，但误差仍然会累积。您可能要处理的其他系统可能具有这么多的非线性 - 这不是夸大其词，只是为了强调一点。处理非线性系统时，您将始终与发散作斗争。

## 算法

您可能急于解决特定问题，并想知道要使用哪种滤波器。我将快速概述可选项。后续章节有一定的独立性，您可以跳过其中的一些章节，但如果您真正想掌握所有材料，我建议您按线性顺序阅读。

非线性滤波器中的主要技术是线性化卡尔曼滤波器和扩展卡尔曼滤波器（EKF）。这两种技术是在Kalman发表论文后不久发明的，自那以后一直是主要使用的技术。飞机的飞行软件、您的汽车或手机上的GPS几乎肯定使用其中一种技术。

然而，这些技术要求极高。EKF在一个点上将微分方程线性化，需要找到偏导数矩阵（雅各比矩阵）的解。这在解析上可能很难或者不可能。如果不可能，你必须使用数值技术来找到雅各比矩阵，但这会增加计算量并引入更多的误差到系统中。最后，如果问题非常非线性，线性化会在每一步引入大量误差，滤波器经常会发散。你不能将一些方程随意放入某个求解器中并期望得到好的结果。这对专业人士来说是一个困难的领域。我注意到大多数卡尔曼滤波教材都只是粗略地涵盖了EKF，尽管它是实际应用中最常用的技术。

最近，这个领域正在以令人兴奋的方式发生变化。首先，计算能力已经增长到我们可以使用曾经超出超级计算机能力的技术。这些技术使用 *蒙特卡罗* 技术 - 计算机生成数千到数万个随机点，并将它们全部与测量结果进行测试。然后基于它们与测量结果匹配的程度，以概率的方式杀死或复制这些点。一个远离测量的点不太可能被保留，而一个非常接近的点则很可能被保留。经过几次迭代后，就会有一堆粒子紧密地跟踪你的对象，以及一个稀疏的点云，在那里没有对象。

这样做有两个好处。首先，该算法即使用于极端非线性问题也非常健壮。其次，该算法可以同时跟踪任意数量的对象 - 一些粒子将匹配一个对象的行为，而其他粒子将匹配其他对象。因此，这种技术通常用于跟踪汽车交通，人群中的人等等。

成本应该是清晰的。对于滤波器中每一步测试成千上万个点在计算上代价昂贵。但现代CPU速度非常快，这是GPU的一个好问题，因为算法的一部分是可并行的。另一个成本是答案不是数学的。使用卡尔曼滤波器时，协方差矩阵为我提供了关于估计误差量的重要信息。而粒子滤波器无法给出一种严格计算这个的方法。最后，滤波器的输出是一堆点；然后我必须弄清楚如何解释它。通常你会做一些像取平均值和标准差的点，但这是一个困难的问题。仍然有许多点不属于被跟踪的对象，因此您首先必须运行某种聚类算法来找到似乎正在跟踪对象的点，然后您需要另一个算法从这些点中产生状态估计。

这是棘手的问题，而且所有计算都相当昂贵。

最后，我们有了一种新的算法，叫做*无香卡尔曼滤波器*（UKF）。它不需要您找到非线性方程的解析解，但几乎总是比EKF表现更好。它在非线性问题上表现良好 - EKF在这些问题上有很大的困难。设计滤波器非常容易。有人会说，UKF仍有争议，但在我看来，UKF在几乎所有方面都比EKF优越。我建议UKF应该是任何实现的起点，特别是如果您不是具有控制理论研究生学位的卡尔曼滤波器专业人士。主要缺点是UKF可能比EKF慢几倍，但这实际上取决于EKF是否通过解析或数值方法解决雅可比矩阵。如果是数值方法，UKF几乎肯定更快。尚未证明（也可能无法证明）UKF始终比EKF产生更准确的结果。但在实践中，UKF几乎总是表现更好。

经常会有很大的影响。它非常容易理解和实现，我强烈建议将这个滤波器作为您的起点。

## 摘要

世界是非线性的，但我们只知道如何解决线性问题。这为卡尔曼滤波器带来了显着困难。我们已经看到了非线性如何以3种不同但等效的方式影响过滤，并简要总结了主要方法：线性化卡尔曼滤波器、扩展卡尔曼滤波器、无迹卡尔曼滤波器和粒子滤波器。

直到最近，线性化卡尔曼滤波器和EKF一直是解决这些问题的标准方法。它们非常难以理解和使用，并且也可能非常不稳定。

最近的进展提供了我认为更好的方法。UKF消除了需要找到偏微分方程解决方案的需要，但通常比EKF更准确。它易于使用和理解。使用FilterPy，我可以在几分钟内启动基本的UKF。粒子滤波器完全摒弃了数学建模，采用了生成数千个点的随机云的蒙特卡罗技术。它运行缓慢，但它可以相对轻松地解决其他棘手的问题。

我收到的关于EKF的电子邮件比其他任何事情都多；我怀疑这是因为大多数书籍、论文和互联网上的论述都使用EKF。如果你的兴趣在于掌握这个领域，当然你会想学习EKF。但是，如果你只是想获得好的结果，我会首先向你指出UKF和粒子滤波器。它们更容易实现、理解和使用，通常比EKF更稳定。

有人可能会对这个建议提出质疑。最近的很多出版物都致力于比较EKF、UKF和其他一些选择的优缺点。您的问题不需要进行类似的比较吗？如果您要发送火箭到火星，那当然需要。您将需要平衡诸如精度、舍入误差、发散、正确性的数学证明以及所需的计算工作量等问题。我无法想象不熟悉EKF的情况。

另一方面，无迹卡尔曼滤波器（UKF）的效果非常出色！我在工作中将其用于实际应用。对于这些应用程序，我大多数时候甚至没有尝试过实现EKF，因为我可以验证UKF可正常工作。在某些情况下，我是否可能从EKF中挤出另外0.2％的性能？当然可以！我在意吗？不！我完全理解UKF的实现，易于测试和验证，我可以将代码传递给其他人，并确信他们可以理解和修改它，而且我不是一个想要与困难方程作斗争的受虐狂，因为我已经有了工作的解决方案。如果UKF或粒子滤波器在某些问题上表现不佳，那么我会转向其他技术，但在此之前不会这样做。而且实际上，UKF通常在广泛的问题和条件范围内提供比EKF更好的性能。如果“非常好”足够好，那么我将花时间解决其他问题。

我这么强调这一点是因为在大多数教科书中，EKF都是重点，而UKF要么根本不提，要么只是简单地介绍两页，让你完全无法使用该滤波器。UKF仍然相对较新，编写新版本的书需要时间。在许多书写的时候，UKF还没有被发现，或者只是一个未经证明但很有前途的奇特现象。但是现在，当我写这篇文章时，UKF已经取得了巨大的成功，并且它需要成为您的工具包中的一部分。这就是我将花费大部分精力尝试教您的内容。

## 参考

<A name="[1]">[1]</A> https://github.com/rlabbe/Kalman-and-Bayesian-Filters-in-Python/blob/master/Supporting_Notebooks/Computing_and_plotting_PDFs.ipynb